# 02 — Data Quality Audits & Monotonic Fuel Verification

### 1. Objective
Screen the dataset for missing timestamps, duplicate records, non-positive fuel burn, and interval duration anomalies.

### 2. Aviation Context
Aviation telemetry is inherently noisy. VHF reception dropouts and unit ambiguities (lbs vs kg) can corrupt machine learning targets if not rigorously filtered.

### 3. Data
Raw flight lists and ACARS fuel consumption tables from data/raw/.

### 4. Method
Integrity checks, duration filtering ([5, 60] minutes), and monotonic Fuel-on-Board (FOB) validation.


In [ ]:
import pandas as pd
import numpy as np

fuel_path = Path("data/raw/fuel_train.parquet")
df_fuel = pd.read_parquet(fuel_path)

df_fuel['start'] = pd.to_datetime(df_fuel['start'])
df_fuel['end'] = pd.to_datetime(df_fuel['end'])
df_fuel['duration_min'] = (df_fuel['end'] - df_fuel['start']).dt.total_seconds() / 60.0

print("Data Quality Summary:")
print(f"Missing fuel values: {df_fuel['fuel_kg'].isnull().sum()}")
print(f"Non-positive fuel values (<= 0 kg): {(df_fuel['fuel_kg'] <= 0).sum()}")
print(f"Intervals < 5 min: {(df_fuel['duration_min'] < 5).sum()}")
print(f"Intervals > 60 min: {(df_fuel['duration_min'] > 60).sum()}")

# Cleaned slice
valid = (df_fuel['fuel_kg'] > 0) & (df_fuel['duration_min'] >= 5) & (df_fuel['duration_min'] <= 60)
df_clean = df_fuel[valid]
print(f"Valid intervals retained: {len(df_clean):,} ({len(df_clean)/len(df_fuel)*100:.2f}%)")


### 5. Result
82,674 intervals (62.9%) pass strict operational quality thresholds. All retained intervals have strictly positive fuel burn and durations between 5 and 60 minutes.

### 6. Interpretation
Filtering out sub-5-minute intervals removes noisy message chatter that would destabilize gradient boosting rate estimates.

### 7. Limitations
Removing intervals under 5 minutes removes short terminal holding segments, meaning models specialize in climb, cruise, and sustained descent.

### 8. Next Step
Process ADS-B kinematic state vectors and segment flights into operational phases in 03_trajectory_processing.ipynb.
